# Logistic Regression Exercise

Now it's your turn to implement logistic regression on a new data set. For this purpose we use the Titanic Dataset. It includes personal information of all passengers on the Titanic as well as they survived the sinking of the Titanic or died.

Here’s the **Data Dictionary** of the dataset:

- PassengerID: type should be integers

- Survived: survived or not

- Pclass: class of Travel of every passenger

- Name: the name of the passenger

- Sex: gender

- Age: age of passengers

- SibSp: No. of siblings/spouse aboard

- Parch: No. of parent/child aboard

- Ticket: Ticket number

- Fare: what Prices they paid

- Cabin: cabin number

- Embarked: the port in which a passenger has embarked.

        - C: Cherbourg , S: Southampton , Q: Queenstown


You will find the data in the data folder.


## What you should do:

- conduct a brief EDA to become familiar with the data
- use Logistic Regression to predict if a passenger died or not

## How to do it:

Time is short, so aim for the simplest viable product first:
1. Load the data

2. Separate features and target 

3. Split the data in train and test

3. Get a quick overview of the train data

4. Agree on a classification metric for the task 

5. Create a simple heuristic/educated guess for the classification first. This is called a "baseline model". It is used to compare more complex models later (in this case: logistic regression). You as a data scientist want to prove how much your work/ML could improve the business metric, therefore you need a baseline model for comparison. In some cases you want to improve on an already existing model in your company which would be your baseline model then. In other cases, there are typical baseline models used in the specific field. For other tasks, you have to come up with a simple but meaningful idea, how to classify the data based on your business understanding (EDA). A baseline model should follow Occam’s Razor principle: "A simple model is the best model". 
    - Example of a baseline model: 
    If the task is to classify cats and dogs, a baseline model could be: We classify every animal as cat if its weight < 5 kg, otherwise the animal is classified as a dog. (The value of 5 kg is an educated guess, based on our business understanding/EDA.) 

6. use one or two already numerical features to create a simple first model
    -  did it even beat your base model?

7. Now you can go through the data science lifecycle again and again:
    - clean the data better

    - get more insights with EDA

    - add more features

    - do feature engineering 
    
    and check if your work improves your model further!

8. Stop whenever time is up or you cannot improve your model any further.

This repository contains a solution to this problem. If you want to compare your final result with the result of this repository solution, choose **25** as random seed and a test size of 30% for your train test split.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score

In [ ]:
%matplotlib inline
plt.rcParams["figure.figsize"] = 10, 8
sns.set_style("whitegrid")

## Get Data

In [ ]:
# Import the dataset
titanic = pd.read_csv("../data/titanic.csv")
titanic.head()

## Define features and Target

In [ ]:
X = titanic.drop(["Survived"], axis=1)
y = titanic["Survived"]

## Train-Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=25, test_size=0.3, stratify=y
)

# Check the shape of the data sets
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

## Exploratory Data Analysis + Featuring Engineering

In [ ]:
# Check data type
print("X_train data type:", type(X_train))
print("y_train data type:", type(y_train))

In [ ]:
# Combine df_X_train and df_y_train into df_train
df_train = pd.concat([X_train, y_train], axis=1, ignore_index=False)
df_train

### Getting a feel for the data

Before we dive into the modelling part we will examine the data.

In [ ]:
# Distribution of target class
df_train["Survived"].value_counts()

In [ ]:
sns.countplot(x="Survived", data=df_train, palette="hls");

In [ ]:
# Missing values
df_train.isnull().sum().sort_values(ascending=False)

In [ ]:
df_train.nunique()

In [ ]:
df_train.info()

Okay, so there are only 623 rows in the titanic data frame. 

Cabin is almost all missing values, so we will drop that variable completely, but what about age? Age seems like a relevant predictor for survival right? We would want to keep the variables, but it has 124 missing values. 

We are going to need to find a way to approximate for those missing values!

#### Dropping missing values: 


So let's just go ahead and drop all the variables that aren't relevant for predicting survival. We should at least keep the following:

- Survived - Since this variable is our target it is obviously relevant.
- Pclass - Does a passenger's class on the boat affect their chance of survival?
- Sex - Could a passenger's gender impact their survival rate?
- Age - Does a person's age impact their survival rate?
- SibSp - Does the number of relatives on the boat (that are siblings or a spouse) affect a person's chance of survival? Could be...
- Parch - Does the number of relatives on the boat (that are children or parents) affect a person's chance of survival? Possible...
- Fare - Does the fare a person paid effect their chance of survival? Maybe - let's keep it.
- Embarked - Does a person's point of embarkation matter? It depends on how the boat was filled... Let's keep it.

What about a person's name, ticket number, and passenger ID number? The passenger ID is a unique number for every passenger and therefore should not contain any useful information for our model. The features name and ticket number might contain helpful information but we would need some proper feature engineering to extract it. For now we will not consider those features for predicting the chance of survival. And as you recall, the cabin variable is almost all missing values, so we will just drop all of these.

In [ ]:
df_train_dr = df_train.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"]).copy()
df_train_dr.head()

In [ ]:
# Display age per class
df_train_dr.groupby(by=["Pclass"])["Age"].describe()

In [ ]:
sns.boxplot(x="Pclass", y="Age", data=df_train_dr, palette="hls");

Speaking roughly, we could say that the younger a passenger is, the more likely it is for them to be in 3rd class. The older a passenger is, the more likely it is for them to be in 1st class. So there is a loose relationship between these variables. So, let's write a function that approximates a passengers age, based on their class. From the box plot, it looks like the median age of 1st class passengers is about 37, 2nd class passengers is 29, and 3rd class passengers is 24.

So let's write a function that finds each null value in the Age variable, and for each null, checks the value of the Pclass and assigns an age value according to the average age of passengers in that class.

In [ ]:
# Check for median age per class
df_train_dr.groupby("Pclass")["Age"].median()

In [ ]:
def age_approx(cols):
    Age = cols["Age"]
    Pclass = cols["Pclass"]

    if pd.isnull(Age):
        if Pclass == 1:
            return 37
        elif Pclass == 2:
            return 29
        else:
            return 24
    else:
        return Age

When we apply the function and check again for null values, we see that there are no more null values in the age variable.

In [ ]:
# Replace missing values in age column
df_train_dr["Age"] = df_train_dr[["Age", "Pclass"]].apply(age_approx, axis=1)
df_train_dr.isnull().sum()

There are 1 null values in the embarked variable. We can drop this 1 records without loosing too much important information from our dataset, so we will do that.

In [ ]:
# Drop rows with missing values in Embarkment column
df_train_dr.dropna(inplace=True)
df_train_dr.isnull().sum()

The next thing we need to do is reformat our variables so that they work with the model.
Specifically, we need to reformat the Sex and Embarked variables into numeric variables.

In [ ]:
# One-hot-encode sex column
gender = pd.get_dummies(df_train_dr["Sex"], drop_first=True, dtype="int")
gender.head()

In [ ]:
# One-hot-encode embarked column
embark_location = pd.get_dummies(df_train_dr["Embarked"], drop_first=True, dtype="int")
embark_location.head()

In [ ]:
# Drop original columns for sex and embarked
df_train_dr.drop(["Sex", "Embarked"], axis=1, inplace=True)
df_train_dr.head()

In [ ]:
# Concatenate one-hot-encoded columns for sex and embarked
df_train_fe = pd.concat([df_train_dr, gender, embark_location], axis=1)
df_train_fe.head()

### Checking for independence between features¶

In [ ]:
# Compute correlations
correlations = df_train_fe.corr()

# Generate a mask for the upper triangle
mask = np.zeros_like(correlations)
mask[np.triu_indices_from(mask)] = True

# Generate a custom diverging colormap
cmap = sns.diverging_palette(220, 10, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(
    correlations,
    mask=mask,
    cmap=cmap,
    vmax=1,
    annot=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.7},
);

In [ ]:
# Fare and Pclass are not independent of each other, so I am going to drop one of these.
df_train_fe.drop(["Fare"], axis=1, inplace=True)
df_train_fe.head()

In [ ]:
# Define features and target variable
X_train_fe = df_train_fe.drop("Survived", axis=1)
y_train_fe = df_train_fe["Survived"]

## Train Model(s) & Metric

Before we start modeling we have to define a metric. In this particular case it is hard to say that making more mistakes with regard to the false positives is worse than having more false negatives and vice versa. Therefore, we will choose a metric that takes precision and recall into account and evaluate our models using the f1-score.

### Baseline Model

Before we start modeling we need to create a baseline model, which is basically an educated guess. 
From the correlation matrix above we can see that the gender is the feature with the highest (in our case negative) correlation to our target variable. Creating a countplot with the gender and survived feature shows also that the majority of the men did not survive, while most of the women survived. As a simple baseline model we could assume that your chance of survival is only related to your gender and whenever you are a man you will inevitably die while all women will survive. That's if course a very drastic view and not completely correct but we only want to create a "baseline" that our more sophisticated models have to beat. 

In [ ]:
df_train_fe.groupby(by=["Survived", "male"])[["male"]].count()

In [ ]:
sns.countplot(x="male", data=df_train_fe, hue="Survived");

In [ ]:
# Defining baseline model
def baseline_model(df: pd.DataFrame):
    """Edugated guess"""
    y_pred = [0 if x == 1 else 1 for x in df.male]
    return y_pred

## Evaluate Model(s)

Before computing prediction with model(s) for the test set we need to apply the same transformation done on the train set
1. drop columns=['PassengerId','Name','Ticket','Cabin']
2. replace missing values in age column with median value from age column in train set
3. drop observations where Embarked column contains nan
4. One-hot-encode sex column
5. One-Hot-encode embarked column
6. Drop original columns for sex and embarked
7. Concatenate one-hot-encoded columns for sex and embarked 

In [ ]:
# Combine X_test and y_test
df_test = pd.concat([X_test, y_test], axis=1, ignore_index=False)

In [ ]:
# 1. Drop columns=['PassengerId','Name','Ticket','Cabin','Fare']
df_test_dr = df_test.drop(
    columns=["PassengerId", "Name", "Ticket", "Cabin", "Fare"]
).copy()

In [ ]:
# 2. Replace missing values in age column with median value from age column in train set
df_test_dr["Age"] = df_test_dr[["Age", "Pclass"]].apply(age_approx, axis=1)

In [ ]:
# 3. drop observations where Embarked column contains nan
df_test_dre = df_test_dr.loc[df_test_dr["Embarked"].notnull(), :].copy()

In [ ]:
# 4. One-hot-encode sex column
gender_test = pd.get_dummies(df_test_dre["Sex"], drop_first=True, dtype="int")

In [ ]:
# 5. One-Hot-encode embarked column
embarked_test = pd.get_dummies(df_test_dre["Embarked"], drop_first=True, dtype="int")

In [ ]:
# 6. Drop original columns for sex and embarked
df_test_dre.drop(["Embarked", "Sex"], axis=1, inplace=True)

In [ ]:
# 7. Concatenate one-hot-encoded columns for sex and embarked
df_test_fe = pd.concat([df_test_dre, gender_test, embarked_test], axis=1)
df_test_fe.head()

In [ ]:
# Define features and target variable
X_test_fe = df_test_fe.drop("Survived", axis=1).copy()
y_test_fe = df_test_fe["Survived"]

In [ ]:
# Compute predictions with baseline model for test set
y_pred_bl_test = baseline_model(X_test_fe)

In [ ]:
# Plot confusion matrix for baseline model
cm_test = confusion_matrix(y_test_fe, y_pred_bl_test)
cm_test

In [ ]:
fig, ax = plt.subplots()
sns.heatmap(
    cm_test,
    cmap="YlGnBu",
    annot=True,
    fmt="d",
    xticklabels=["Not Survived", "Survived"],
    yticklabels=["Not Survived", "Survived"],
    ax=ax,
)

fig.supxlabel("predicted")
fig.supylabel("actual");

In [ ]:
# Calculate f1-score for baseline model
print("F1-score test: ", round(f1_score(y_test_fe, y_pred_bl_test), 2))

Our baseline model manages to 
+ 28 out of the 165 Not Survived observations get misclassified as Survived, and the remaining 137 are correctly classified
+ 31 out of the 102 Survived observations get misclassified as Not Survived, and the remaining 71 are correctly classified 

The computed f1-score is 0.71. 

Let's see if we can improve this with a logistic regression. 

## Logistic Regression

In [ ]:
# Instantiate model and fit it on train data
logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train_fe, y_train_fe)

In [ ]:
# Make predictions for test set
y_pred_log_test = logreg.predict(X_test_fe)

In [ ]:
cm_log_test = confusion_matrix(y_test_fe, y_pred_log_test)
cm_log_test

In [ ]:
fig, ax = plt.subplots()
sns.heatmap(
    cm_log_test,
    cmap="YlGnBu",
    annot=True,
    fmt="d",
    xticklabels=["Not Survived", "Survived"],
    yticklabels=["Not Survived", "Survived"],
    ax=ax,
)

fig.supxlabel("predicted")
fig.supylabel("actual");

In [ ]:
# Calculate f1-score for logistic regression model
print("F1-score: ", round(f1_score(y_test_fe, y_pred_log_test), 2))

The confusion matrix shows that 124 and 84 are the number of correct predictions. 41 instances from class 0 (Not Survived) where incorrectly classified as belonging to class 1 (Survived). With regard to the false negatives, the instances that actually belong to class 1 but where classified as being class 0, our model improved quite a bit compared to the baseline. 

Also the f1-score is with 0.74 slightly higher compared to the baseline model, but I would probably not call it a huge improvement. Seems that our choice of a baseline model was not that bad...


In [ ]:
# Print classification report for more information
print(classification_report(y_test_fe, y_pred_log_test))

**Let us learn about [Classification Report](https://muthu.co/understanding-the-classification-report-in-sklearn/)**